For det er muligt at lave markov kæderne, så skal der importers nogen bibloteker.

In [68]:
import re

In [2]:
import numpy as np


In [3]:
import random

Den første del før, man kan anvende noget som helst omkring Markov kæder er at lave selve Markov kæden altså for skaffet dataen.
Det gøre vi først ved at læse noget data omtil tokens

In [72]:

def read_file_into_tokens(file_path):
    """
    Læser en given fil, og omdanner det til en rå liste af tokens (altså individuelle ord)
    """

    # Her læser vi filens indhold
    file_contents = ""
    file = open(file_path, "r", encoding="utf-8")
    for line in file:
        file_contents += line
    file.close()
    
    # Vi laver alt indhold lower-case
    file_contents = file_contents.lower()
    
    # Her bruger vi et værktøj der hedder regex.
    # Det er bare en smart måde at vi hurtigt kan få omdannet vores fil-indhold til en liste af strings ved at opdele den ved specifikke tegn
    raw_tokens = re.split(r"(?![:,\.;\"][0-9])[:,\.;?\" \r\n]+", file_contents)
    
    # Behold kun de tokens som har en værdi (altså ingen tokens som er tom tekst)
    tokens = []
    for raw_token in raw_tokens:
        if raw_token:
            tokens.append(raw_token)
    return tokens


In [73]:
exsample_tokens = read_file_into_tokens(r"C:\UNF\KIC2026\GitCode\KIC26\Markov_Chain_kode\test_data_file.txt")

Efter vi har lavet tokens, så skal vi omdanne dem til states i Markov kæder, og for det så har vi brug for tokense vi lavet ligefør.

In [74]:

def create_states(data, amount_of_words=1):
    """
    Skaber individuelle states til vores data ud fra mængden af ord der betragtes på en gang  
    """
    list_of_states = []
    
    # Vi gå gennem vores data ud fra index idet vi skal kombinere dele af listen af gangen
    for i in range(len(data) + 1 - amount_of_words):
        # Kombiner fremliggende naboer op til mængden af betragtede ord (eg. amount_of_words = 2 medfører at hvert token bliver 2 ord lang)
        list_of_states.append(" ".join(data[i:i + amount_of_words]))

    return list_of_states

In [75]:
exsample_states=create_states(exsample_tokens)

Når statesne er lavet alt der, så er det bare selve makrov kæden der mangler at blive lavet.

In [76]:

def generate_markov_chain(states):
    markov_chain={}
    """
    Genererer en dictionary som angiver mængden af forekomster og naboliggende ord (og deres hyppigheder) i vores datasæt som en 
    """
    for index, word in enumerate(states):
        #Positionen til det næste ord i vores liste
        nabo_index = index + 1

        # Hvis det næste ord i sekvensen ligger udenfor listen, er vi færdige.
        if nabo_index >= len(states):
            # Hvis det allersidste ord ikke findes andre steder end i slutningen, kan det som state i markov kæden ikke føre til andre states.
            # I dette tilfælde, siger vi derfor blot at dens næste ord er sig selv, således at vi ikke har en slags 'blind vej' i vores endelige markov kæde.
            if word not in markov_chain:
                markov_chain[word] = [1,{}]
                #markov_chain[word] = [1,{}]
                markov_chain[word][1][word] = 1
                markov_chain[word][1][next_word] = 1
            break
            
        next_word = states[nabo_index].split(" ")[-1]
        
        # Hvis vi har registreret vores ord i dictionary i forvejen, så tilføj 1 til hyppigheden og forekomsterne
        if word in markov_chain:
            # Ved 0 indekset ligger totale mængde forekomster i teksten
            markov_chain[word][0] += 1
            
            # Ved 1 indekset ligger en dictionary som viser hyppigheden for hvert naboliggende ord.
            # Opdater hyppigheden for det naboliggende ord 
            if next_word in markov_chain[word][1]:
                markov_chain[word][1][next_word] += 1
            else:
                markov_chain[word][1][next_word] = 1

        #Hvis ordet ikke er registreret endnu, registrerer vi den til vores data dictionary
        else:
            markov_chain[word] = [1,{}]
            markov_chain[word][1][next_word] = 1
    
    # Ved alle states sorterer vi de næste states ud fra deres hyppighed. 
    # Dette gør vi for at gøre det nemmere at beregne sandsynlighed
    for state in markov_chain.keys():
        markov_chain[state][1] = dict(sorted(markov_chain[state][1].items(), key = lambda item: item[1], reverse=True))

    return markov_chain


In [77]:
exsample_markov_chain=generate_markov_chain(exsample_states)


Nu er markov kæden færdig, så er den klar til at produccere text. Når man skal anvende den, så kræver det ud over sin markov kæde at have både et start ord, der er hvilket state markov kæden starter på, men også en temperature, som er hvor tilfældig den skal være 0 er full determinitisk, 1 er full markov kædde, hvor 2 er full tilfældighed.
Start ordet skal være i makrkov kæden.

In [ ]:
def calculate_state_chances(temperature, total_occurences, possible_next_states):
    """
    Beregner chancerne for at gå til hvert mulige state
    """
    return [calculate_single_state_chance(temperature, int(total_occurences), len(possible_next_states), state) for state in possible_next_states.values()]


def calculate_single_state_chance(temperature, total_occurances, possible_next_state_count, next_state_frequency):
    """
    Beregner en chance for at gå til hvert mulige state
    """
    
    if temperature<=1:
        # Denne beregning gør at temperatur på 0 medfører en determinisk generation, samt at ved 1 så generering chancen direkte ækvivalent med frekvensen / totale hyppighed.
        # Imellem disse 2 ender, er en glat overgang.
        return  next_state_frequency / ((1-temperature) * next_state_frequency + total_occurances * temperature)
    
    else:
        
        # Hvis temperature = 1, bliver den flippet til 0. Derefter beregnes
        """
        Først temperature = temperature-1 så temperature = 0
        I slutning skal det være next_state_frequency/totalt_occurences
        så next_state_frequency*(1-temperature)+temperature = next_state_frequency*(1-0)+0 = next_state_frequency
        så temperature = total_occurrences

        q= dif_amount*r+total_amount*(1-r) = dif_amount*0+total_amount*(1-0) =total_amount
        q = total_amount

        t/q =n/total_amount

        """
        
        
        # Hvis r=2 så bliver den 1 og det her er, hvad der skal ske med den.
        """
        Først r = r-1 så r = 1
        I slutning skal det være 1/dif_amount
        så n*(1-r)+r =n*(1-1)+1 = n*0+1 = 1
        så t = 1

        q= dif_amount*r+total_amount*(1-r) = dif_amount*1+total_amount*(1-1) =dif_amount
        q = dif_amount
        t/q= 1/dif_amount 
        """
        
        temperature = temperature-1
        t = next_state_frequency*(1-temperature)+temperature

        q = possible_next_state_count*temperature+total_occurances*(1-temperature) 
        return t/q
     

def generate_text(markov_chain, amount_to_generate = 10, starting_state = None, temperature = 1):
    """
    Genererer tekst ud fra den givne markov kæde
    """
    
    # Hvis vi ikke har givet et start state, så vælg et tilfældigt
    if starting_state is None:
        starting_state = random.choice(markov_chain.keys())
    
    # Gør vores start til lower-case
    starting_state = starting_state.lower()

    # Tjek vores input
    if starting_state not in markov_chain.keys():
        raise ValueError("Start staten findes ikke som et state i markov kæden")
    if (amount_to_generate <= 0):
        raise ValueError("Du kan ikke generere nul eller en negativ mængde tekst")
    if (temperature < 0 or temperature > 2):
        raise ValueError("Temperatur skal være mellem 0 og 2")
    
    # Vi bestemmer mængden af ord betragtet på en gang ud fra det givne start ordet. 
    # Dette bør altid være den samme mængde som mængden af ord betragtet i selve markov kæden.
    amount_of_words = len(starting_state.split(" "))
    
    # Denne liste vil indeholde vores genererede tekst. Den starter derfor med at indeholde vores startord
    generated_text = starting_state.split(" ")
    
    # Vi bruger denne til at holde styr på hvad vi gerne vil generere det næste ud fra
    prev_state = starting_state
    
    # Generer tekst
    for i in range(amount_to_generate):
        total_occurances = markov_chain[prev_state][0]
        next_possible_states = markov_chain[prev_state][1]
        next_possible_state_chances = calculate_state_chances(temperature, total_occurances, next_possible_states)
        
        # Vi opsummere løbende chancerne for hvert state. Når den overgår vores tilfældigt-satte threshold, er det den state vi går til næst.
        # Dette er den simpleste måde at programmatisk vælge et element fra en liste tilfældigt ud fra en chance.
        random_threshold = np.random.rand(1,1)[0][0]
        sum_chance = 0

        # Variablen er, hvor meget chance der er tilbage før der har noget på, som bliver anvendt senere til en form for normalisering
        amount_left = 1

        # Variablen er hvad der skal ganges med, så det er normaliset på en måde for, at sikre at det altid bliver summed up til 1 i slutning
        multiply_to_norm = 1
        
        # Vi gennemgår hvert næste mulige state med et indeks, idet vi bruger seperate lister til chancerne og selve states.
        for next_state_index, next_state_text in enumerate(next_possible_states.keys()):
            
            # Tjek om vores tilfældige threshold er nået- hvis ja, er det nuværende element det tilfældigt valgte. Ellers, fortsæt gennem vores muligheder.
            sum_chance += next_possible_state_chances[next_state_index]*multiply_to_norm
            if sum_chance <= random_threshold:
                # Opdater mængden der er left baseret på sum_chancen
                amount_left = 1 - sum_chance 
                #Den bliver til hvad man skal gange alle værdierne med, så deres sum giver mængden der er tilbage for at sikre at der altid vil 1.
                multiply_to_norm = 1/(sum(next_possible_state_chances[next_state_index:])*(1/(amount_left)))
                continue

            # Vi har hermed vores næste state.
            generated_text.append(next_state_text)
            
            # Opdater det nuværende state til at være de sidste ord i vores genererede tekst
            prev_state = " ".join(generated_text[-amount_of_words:])
            break

    # Vores genererede tekst består af individuelle ord. Vi samler dem for at få et sammenhængende stykke tekst
    return " ".join(generated_text)


In [87]:
generate_text(markov_chain=exsample_markov_chain,amount_to_generate=15,starting_state="hej",temperature=1)

'hej med dig mit navn er john og mit navn er john og mit'

Som der kan ses, så er teksten der bliver generet ikke alt for god, som skyldes at at datasettede er meget lille og derfor har den ikke så meget den har trænet på.

Selvom det er dårligt ligenu, så er det en metode at generer tekst.

Med udgangpunkt i methoden vist ovenover, så skal i kigge på forskellige måder, at ændre outputted. I de følgende opgaver er der, somså ikke et rigtig svar men mere generelle svar.

Opgave 0)

Ligesom der er blevet gjort ovenover til at generer tekst skal i anvende samme methode til det.

Vælg et dataset fill

Ud fra datasettet lav tokens

ud fra tokense lav en markov kæde

Ud fra markov kæden lav noget tekst med forskellige starting states(hvis du ikke med hvilken nogen starting states du kan vælge kan du altid kigge i filen).

Lav du en funktion, som ud fra en fil laver en markov kæde.

Anvend funktion til at lave forskellige markov kæder ud fra forskellige filer.

Anvend de forskellige markov kæder du har lavet, og se hvor forskellige outputtede bliver.


In [ ]:
#Lav de forskellige trin her

In [ ]:
#Lav tekst her

In [ ]:
#Lav markov kæde funktionen her

def make_markov_chain_from_file(file_path):
    #implementer funktionen her
    raise AssertionError("Funktion not implementet")

In [ ]:
#Lav tekst her

Vælg en af markov kæderne, som de lavet i opgave 0 til at de næste opgaver.

Opgave 1)

Ændre randomness til værdier mellem 0 og 2, og se hvad det sker. 

Hvilken værdi er den bedste?

Hvad sker der når den er 0?

Hvad sker der når den 2?

In [80]:
#anvend generete_tex med jeres markov kæde her få de forskellige svar

Opgave 2)

I opgave 1) skulle i sætte randomnes til 0, og når i gjorde det vil i har set, at den loopede, men hvorfor loopede den?

Opgave 3)

I har set, hvad der sker hvis man ændre randomness, men hvad hvis i ændrede formellen på hvordan den beregnede sandynlighederne, og se hvad det gøre ved outputtede. Funktion calculate_single_state_chance er den funktion, som i større trek beregner de forskellige sandynligheder(der er et par andre ting, som også bliver taget med). 

Ændre på calculate_single_state_chance så meget eller så lidt som i har lyst til


Lave opgave 1) med den nye chance funktion
 

In [ ]:
#Her er hvor i skal skrive jeres nye chance_formel 
"""
Funktionen nede under er chance funktion i skal, som i skal ændre på.
Den skal tage de samme variavbler, som den gamle, men i vælger selv hvad der sker, når temperaturen ændre sig.
"""

def calculate_single_state_chance(temperature, total_occurances, possible_next_state_count, next_state_frequency):
    """
    Beregner en chance for at gå til hvert mulige state
    """
    
    if temperature<=1:
        # Denne beregning gør at temperatur på 0 medfører en determinisk generation, samt at ved 1 så generering chancen direkte ækvivalent med frekvensen / totale hyppighed.
        # Imellem disse 2 ender, er en glat overgang.
        return  next_state_frequency / ((1-temperature) * next_state_frequency + total_occurances * temperature)
    
    else:
        
        # Hvis temperature = 1, bliver den flippet til 0. Derefter beregnes
        """
        Først temperature = temperature-1 så temperature = 0
        I slutning skal det være next_state_frequency/totalt_occurences
        så next_state_frequency*(1-temperature)+temperature = next_state_frequency*(1-0)+0 = next_state_frequency
        så temperature = total_occurrences

        q= dif_amount*r+total_amount*(1-r) = dif_amount*0+total_amount*(1-0) =total_amount
        q = total_amount

        t/q =n/total_amount

        """
        
        
        # Hvis r=2 så bliver den 1 og det her er, hvad der skal ske med den.
        """
        Først r = r-1 så r = 1
        I slutning skal det være 1/dif_amount
        så n*(1-r)+r =n*(1-1)+1 = n*0+1 = 1
        så t = 1

        q= dif_amount*r+total_amount*(1-r) = dif_amount*1+total_amount*(1-1) =dif_amount
        q = dif_amount
        t/q= 1/dif_amount 
        """
        
        temperature = temperature-1
        t = next_state_frequency*(1-temperature)+temperature

        q = possible_next_state_count*temperature+total_occurances*(1-temperature) 
        return t/q
     

In [ ]:
#Her laver i opgave 1 med den nye funktion

Opgave 4)

Filen arrangoererne.txt er de sammlede tekster, som der er blevet sat ind i filen. I skal nu lave en markov kæde ud fra den fil.

Genere noget tekst med markov kæden.

Filen arrangoerne_copy.txt skal du nu tilføje ting til filen, så når du laver tekst med markov kæden fra den, så skal den, som minimum sige at du er ar arrangøre på campen(men du kan sagtens også se om du kan få den til at sige andre ting).


In [ ]:
#Lav opgaven omrking den oprindlige tekst med arngørerne her

In [ ]:
#Lav opgaven omrking den modificeren arngøre tekst her

Opgave 4)

I opgave 4 skal du kigge på hvad der sker, når man anvender en markov kæde til at lave sin data for sig.

Vælg en markov kæde, som du har lavet i en af de forrige opgaver.

Før det er man kan lave en ny markov kæde, så skal man lave selve dataen. Til at lave den nye data skal i anvende funktion "generate_new_data". Funktion vil lalve en fil baseret på den tidliger markov kæde, lidt ligesom når i anvendt "generate_text" tidliger. Funktion tager følgende argumenter:
markov_chain: Selve markov kæden
new_file_name: Navnet på fillen
amount_of_words: Antal ord den skal generer i fillen
temperature: Temperaturen der skal anavendes
amount_of_times: Antal gange den laver ny data ud fra markov kæden
start_state: Er start staten der bliver anvendt når den skal lave dataen

Prøv nu at anvende funktion til at lave en data fill ud fra en markov kæde hvor amount_of_times = 1.

I fillen vil der være "samme" tekst, som hvis du anvendt "generate_text".

Prøv nu at tage den fil du har generet, og lav en markov kæde ud fra fillen for at se hvordan teksten bliver.


I stedet for at skulle lave det til en data fil først, så kan i også anvende funktionen "generate_new_markov_chain", som tager samme alle de samme argumenter ud over new_file_name siden den laver direkte en markov kæde.

Anvend "generate_new_markov_chain"(kan også godt bruge "generate_new_data" det er bare letter at anvende "generate_new_markov_chain") til at løse de næste opgaver.

Ændre kun på amount_of_words og se hvad der sker.

Ændre kun på temperature og se hvad der sker.

Ændre kun på start_state og se hvad der sker.

Ændre kun på amount_of_times og se hvad der sker.

Prøv nu at ændre på flere af argumenter på en gang og se hvad der sker med teksten.

In [ ]:



# Funktion her tager en markov kæde, og itererer på den ved at den skabe ny text, generere en ny markov kæde ud fra den tekst, og så gentag
# Det er lidt ligesom de der "google translated ___ times" videoer
"""
amount_of_words er hvor mange ord der skal være i den nye fil
amount_of_times er antal af gange der laver ny data ud fra markov kæden, så amount_of_times = 1 så er det direkte fra den oprindlige markov kæde,
hvis amount_of_times = 2 så laver den dataen ud fra en markov kæde, som er baseret på den fil der kommer, hvis amount_of_times = 1
"""
def generate_iterated_markov_chain(markov_chain,amount_of_words=10000, temperature=1, amount_of_times=1, start_state = None):
    new_markov_chain = markov_chain
    
    # Funktion looper, så den iterativt laver en ny markov kæde efter hvert loop
    for i in range(amount_of_times):
        print("",f"Iteration {i}")
        
        # Vi genererer først ny text med den nuværende markov kæde
        text = generate_text(new_markov_chain, amount_of_words, start_state, temperature).split(" ")
        
        # Vi genererer nu en ny markov kæde med vores nyligt genererede tekst
        new_markov_chain = generate_markov_chain(create_states(text))

    return new_markov_chain

In [ ]:
#Lav opgaverne med generate_new_data her

In [ ]:
#lav opgaverne med generate_new_markov_chain her

Endtil videre har vi kun kigget på, hvor start state havde et ord, men der er enten der stopper os for at gøre så hver state har flere ord end et. Når vi laver statesne, så skal man gøre så at den tager n mængde ord i stedet for 1. I funktion create_states så er der et argument, som hedder amount_of_words, som styrer hvor mange ord der skal være på hver state. 

Opgave 5)

Ændre på funktion du lavede i opgave 1, så den nu også tager et argument på hvor mange ord der skal være på de forskellige states.

Vælg et dataset.

Lav en markov kæde ud fra datasettet, men med to ord som states og se, hvordan den generer tekst i forhold til en.

Lav nu en markov kæde med 3 ord, og se hvordand en generer.


In [ ]:
#Lav den nye funktion her

In [ ]:
#Generere markov kæderne og teksterne her

Opgave 6)

I har ændret på mange forskellige parameter, og fået forskellige resultater. Lave nu den bedste markov kæder tekst generator du kan ved at ændre på de forskellige argumenter.

In [ ]:
#Jeres bedste markov kæde tekst generetor

Opgave 7)

Senere skal i arbejde med LLM, hvor i skal anvende et andre dataset, som ligger i mappen DataTextHistorie anvend de dataset til at genere text, så i vil har noget at sammenligne med når i anvender LLM til det.


In [ ]:
#Lav markov kæder med den nye data her